# 전체 전처리 + A/B 가능성 점검

이 노트북은 두 가지를 한 번에 다룬다.

1. **B 가능성 점검(전처리 포함)**: 22종 데이터셋 전체에 대해 SMILES 파싱, 부모 분자 확정(RDKit LargestFragmentChooser), 정준 호변이성질체 계산, Bemis-Murcko 골격, 호변이성질체/양성자화 상태/염 형태/입체 표기 카운트를 수행한다. `scripts/eda_and_prescreen.py`를 그대로 불러와 쓰므로 로컬 실행 결과와 100% 동일한 로직이다.
2. **A 가능성 점검(신규)**: SMILES 언어모델(ChemBERTa 계열)을 실제로 미세조정해서 되는지, 얼마나 걸리는지 확인하고, 학습된 모델과 무작위 초기화 모델의 등가 SMILES 예측 분산(A)을 비교한다. 지금까지 한 번도 실행해본 적 없는 부분이다.

실행 전에 런타임을 GPU로 바꾼다: 상단 메뉴 `런타임 -> 런타임 유형 변경 -> GPU`.

참고 문서: `docs/연구계획서.md` 5.3~5.5절(전처리·모델 구성), 5.4절(사전 선별), `docs/실행계획.md` 2절(게이트 운영), `docs/문헌_정리.md`(canSAR, TTA-XAI 논문).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/Conference_2026'
assert os.path.isdir(PROJECT_ROOT), (
    f"{PROJECT_ROOT} 를 찾을 수 없다. Drive에서 폴더 이름이 다르면 이 경로를 직접 수정한다."
)
print('project root:', PROJECT_ROOT)

In [ ]:
# CLAUDE.md 방침: Colab에서는 로컬 venv를 쓰지 않고 노트북 안에서 그때그때 설치한다.
!pip install -q rdkit dimorphite-dl transformers accelerate

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')

if not os.path.isdir(RAW_DIR) or not os.listdir(RAW_DIR):
    print('data/raw가 비어 있다. 데이터셋을 먼저 내려받는다.')
    %cd {PROJECT_ROOT}
    !python scripts/download_admet.py
    !python scripts/download_esol_pilot.py
else:
    print('data/raw 확인됨:', sorted(os.listdir(RAW_DIR))[:5], '...')

## 1부. B 가능성 점검 및 전처리 (22종 전체)

`scripts/eda_and_prescreen.py`의 로직을 그대로 불러온다. 부모 분자 추출은 RDKit `LargestFragmentChooser(preferOrganic=True)`를 쓰고, 중복 판정은 정준 호변이성질체(`TautomerEnumerator.Canonicalize`, canSARchem_RDKit과 동일 설정) 기준으로 한다. 상세 이유는 `docs/문헌_정리.md`의 canSAR 항목 참고.

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)

import time
from pathlib import Path

import pandas as pd
from scripts.eda_and_prescreen import load_dataset, profile_dataset

PROCESSED_DIR = Path(PROJECT_ROOT) / 'data' / 'processed'
RAW_DIR_PATH = Path(PROJECT_ROOT) / 'data' / 'raw'

In [ ]:
# 이미 처리된 데이터셋은 건너뛴다 (data/processed/<이름>/molecule_profile.csv 존재 여부로 판단).
# 로컬에서 이미 22종을 전부 처리했다면 이 셀은 전부 skip으로 끝난다.
# 부모 분자 추출 로직을 바꾼 뒤 다시 돌리고 싶으면, 먼저 data/processed/<이름>/
# 폴더를 지우고 이 셀을 실행한다.

names = sorted(
    p.name for p in RAW_DIR_PATH.iterdir()
    if p.is_dir() and p.name not in ('_tdc_cache', 'esol_pilot')
)

for name in names:
    cache_path = PROCESSED_DIR / name / 'molecule_profile.csv'
    if cache_path.exists():
        print(f'[skip, cached] {name}')
        continue
    t0 = time.time()
    s = profile_dataset(name)
    print(f"[{time.time()-t0:>6.1f}s] {name:35s} n={s['n_total']:>6d} "
          f"salt={s['pct_with_salt']}%  stereo={s['pct_with_stereo']}%  "
          f"multi-taut={s['pct_multi_tautomer']}%  multi-proto={s['pct_multi_protomer']}%")

print('\n22종 전처리 완료 (신규 처리 또는 캐시 확인).')

In [ ]:
# 5.4절 게이트 판정: 각 축이 10% 미만이면 그 데이터셋에서는 주 분석 제외.
rows = []
for name in names:
    detail_path = PROCESSED_DIR / name / 'molecule_profile.csv'
    if not detail_path.exists():
        continue
    d = pd.read_csv(detail_path)
    valid = d[d['valid']]
    n = len(valid)
    rows.append({
        'dataset': name,
        'n_valid': n,
        'B1_미소상태': round(100 * ((valid['n_tautomers'] >= 2) | (valid['n_protomers'] >= 2)).mean(), 1) if n else None,
        'B2_염형태': round(100 * valid['has_salt'].mean(), 1) if n else None,
        'B3_입체표기': round(100 * valid['has_stereo'].mean(), 1) if n else None,
    })

gate_df = pd.DataFrame(rows).sort_values('dataset')
gate_df['B1_생존'] = gate_df['B1_미소상태'] >= 10
gate_df['B2_생존'] = gate_df['B2_염형태'] >= 10
gate_df['B3_생존'] = gate_df['B3_입체표기'] >= 10

gate_df.to_csv(PROCESSED_DIR / 'gate_decision_final.csv', index=False)
print(f"B1 생존: {gate_df['B1_생존'].sum()}/{len(gate_df)}   "
      f"B2 생존: {gate_df['B2_생존'].sum()}/{len(gate_df)}   "
      f"B3 생존: {gate_df['B3_생존'].sum()}/{len(gate_df)}")
gate_df

## 2부. A 가능성 점검 (SMILES 언어모델 미세조정)

5.5절: 표현 불안정성(A)은 SMILES를 직접 읽는 모델이 있어야만 0이 아닌 값을 가진다. 지금까지 한 번도 실행해본 적이 없으므로, 여기서 세 가지를 확인한다.

1. 미세조정이 실제로 되는가 (에러 없이 수렴하는가)
2. 얼마나 걸리는가 (전체 계산 예산 산정용, 실행계획.md 2절 네 번째 게이트 항목)
3. 학습된 모델과 무작위 초기화 모델의 A가 실제로 다른가 (TTA-XAI 논문의 경고 -- 무작위 모델도 비슷한 분산을 보일 수 있다는 -- 를 우리 데이터로 직접 검증)

심층 분석 대상 하나(작은 데이터셋)로 시험한다. 기본값은 `dili`(475개, 분류).

In [ ]:
import numpy as np
import torch
from rdkit import Chem
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cpu':
    print('경고: GPU가 아니다. 런타임 유형을 GPU로 바꾸는 것을 권한다 (상단 메뉴).')

TARGET_DATASET = 'dili'
MODEL_NAME = 'DeepChem/ChemBERTa-10M-MTR'
N_EPOCHS = 5
BATCH_SIZE = 16
MAX_LEN = 128

In [ ]:
df = load_dataset(TARGET_DATASET)
print(f'{TARGET_DATASET}: n={len(df)}, 라벨 분포:')
print(df['Y'].value_counts(normalize=True))

train_df, val_df = train_test_split(df, test_size=0.2, random_state=0, stratify=df['Y'])
print(f'train={len(train_df)}  val={len(val_df)}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(smiles_list):
    return tokenizer(list(smiles_list), padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors='pt')

train_enc = encode(train_df['Drug'])
train_labels = torch.tensor(train_df['Y'].values, dtype=torch.long)
val_enc = encode(val_df['Drug'])
val_labels = torch.tensor(val_df['Y'].values, dtype=torch.long)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
optim = torch.optim.AdamW(model.parameters(), lr=2e-5)

n_train = len(train_df)
t0 = time.time()
model.train()
for epoch in range(N_EPOCHS):
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    for start in range(0, n_train, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        batch = {k: v[idx].to(DEVICE) for k, v in train_enc.items()}
        labels = train_labels[idx].to(DEVICE)
        optim.zero_grad()
        out = model(**batch, labels=labels)
        out.loss.backward()
        optim.step()
        epoch_loss += out.loss.item() * len(idx)
    print(f'epoch {epoch}: loss={epoch_loss/n_train:.4f}')

finetune_seconds = time.time() - t0
print(f'\n미세조정 소요 시간: {finetune_seconds:.1f}초 ({TARGET_DATASET}, n={n_train}, {N_EPOCHS} epoch, {DEVICE})')

In [ ]:
model.eval()
with torch.no_grad():
    val_batch = {k: v.to(DEVICE) for k, v in val_enc.items()}
    val_logits = model(**val_batch).logits
    val_pred = val_logits.argmax(dim=-1).cpu().numpy()

val_acc = (val_pred == val_labels.numpy()).mean()
print(f'검증 정확도: {val_acc:.3f} (모델이 실제로 뭔가를 배웠는지 확인용, 성능 목표치는 아님)')

### A 신호 계산: 학습된 모델 vs 무작위 초기화 모델

TTA-XAI 논문(Hartog et al. 2024, `문헌_정리.md` 참고)의 경고를 우리 데이터로 직접 확인한다. 검증 집합에서 분자 여러 개를 뽑아, 각각 등가 SMILES 5개를 생성하고 예측 확률의 표준편차를 잰다. 학습된 모델과 인코더까지 무작위로 다시 초기화한 모델을 같은 방식으로 비교한다.

In [ ]:
def random_smiles_variants(smiles, n=5, max_tries=15):
    mol = Chem.MolFromSmiles(smiles)
    out = set()
    for _ in range(max_tries):
        out.add(Chem.MolToSmiles(mol, canonical=False, doRandom=True))
        if len(out) >= n:
            break
    return list(out) if out else [smiles]


def compute_A_for_model(model, molecules, n_variants=5):
    """분자 목록에 대해 등가 SMILES 예측 표준편차(A)를 각각 구해 배열로 반환."""
    model.eval()
    a_values = []
    for smi in molecules:
        variants = random_smiles_variants(smi, n=n_variants)
        enc = encode(variants)
        with torch.no_grad():
            batch = {k: v.to(DEVICE) for k, v in enc.items()}
            probs = torch.softmax(model(**batch).logits, dim=-1)[:, 1].cpu().numpy()
        a_values.append(probs.std())
    return np.array(a_values)

In [ ]:
N_PROBE_MOLECULES = 30
probe_smiles = val_df['Drug'].sample(n=min(N_PROBE_MOLECULES, len(val_df)), random_state=1).tolist()

print('학습된 모델의 A 계산 중...')
a_trained = compute_A_for_model(model, probe_smiles)

print('무작위 초기화 모델 준비 중 (인코더까지 재초기화)...')
fresh_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
for module in fresh_model.modules():
    if hasattr(module, 'reset_parameters'):
        module.reset_parameters()

print('무작위 모델의 A 계산 중...')
a_random = compute_A_for_model(fresh_model, probe_smiles)

print(f'\n학습된 모델   A 평균={a_trained.mean():.4f}  중앙값={np.median(a_trained):.4f}')
print(f'무작위 모델   A 평균={a_random.mean():.4f}  중앙값={np.median(a_random):.4f}')
print(f'\n비율(학습됨/무작위): {a_trained.mean() / max(a_random.mean(), 1e-8):.2f}')
print('1에 가까우면 TTA-XAI 논문의 경고(A가 토큰화 아티팩트일 수 있다)가 우리 데이터에서도 재현된 것.')
print('학습된 모델의 A가 뚜렷이 작으면(비율이 1보다 충분히 작으면), 미세조정이 A를 안정화한다는 근거.')

In [ ]:
import json

a_feasibility_summary = {
    'dataset': TARGET_DATASET,
    'model': MODEL_NAME,
    'device': DEVICE,
    'n_train': n_train,
    'n_epochs': N_EPOCHS,
    'finetune_seconds': round(finetune_seconds, 1),
    'val_accuracy': round(float(val_acc), 4),
    'n_probe_molecules': len(probe_smiles),
    'A_trained_mean': round(float(a_trained.mean()), 5),
    'A_trained_median': round(float(np.median(a_trained)), 5),
    'A_random_mean': round(float(a_random.mean()), 5),
    'A_random_median': round(float(np.median(a_random)), 5),
    'ratio_trained_over_random': round(float(a_trained.mean() / max(a_random.mean(), 1e-8)), 3),
}

out_path = PROCESSED_DIR / 'a_axis_feasibility.json'
with open(out_path, 'w') as f:
    json.dump(a_feasibility_summary, f, ensure_ascii=False, indent=2)

print(f'저장: {out_path}')
a_feasibility_summary

## 결론 읽는 법

- `finetune_seconds`: 실행계획.md 2절 게이트의 네 번째 항목. 이 값을 22종에서 몇 개까지 심층 분석할 수 있는지(4~6종) 계산하는 데 쓴다.
- `val_accuracy`: 너무 낮으면(예: 0.5 근처, 이진 분류 기준) 모델이 사실상 아무것도 못 배운 것이므로 A 비교 결과 자체를 신뢰하기 어렵다. 이 경우 에폭 수를 늘리거나 학습률을 조정해서 다시 시도해야 한다.
- `ratio_trained_over_random`: 1에 가까우면 TTA-XAI 논문이 경고한 대로 A가 토큰화 아티팩트일 가능성이 있다는 뜻이고, 이 경우 연구계획서 4.4절의 2차 가설(A) 검증에 더 신중해야 한다. 뚜렷이 1보다 작으면 미세조정이 실제로 표현을 안정화한다는 긍정적 근거다.

이 결과를 가지고 `docs/실행계획.md`의 게이트 판정과 `docs/연구계획서.md` 5.2절(심층 분석 대상 확정)을 업데이트한다.